In [50]:
from pathlib import Path
from collections import defaultdict
from typing import Callable

import math
import re
import sys
import unicodedata
import xml.etree.ElementTree as ET

import pandas as pd

from nltk.stem import SnowballStemmer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

### Leemos los documentos

In [51]:
DOCS_PATH = Path("docs-raw-texts")
QUERIES_PATH = Path("queries-raw-texts")
RELEVANCE_PATH = Path("relevance-judgments.tsv")

In [52]:
def read_naf(path: Path) -> str:
    root = ET.parse(path).getroot()

    title = ""
    raw = ""

    for elem in root.iter():
        tag = elem.tag.split("}")[-1]

        if tag == "fileDesc":
            title = elem.attrib.get("title", "")

        elif tag == "raw":
            raw = elem.text or ""

    return f"{title} {raw}".strip()

def get_id(path: Path) -> str:
    return path.stem.replace("wes2015.", "")

In [53]:
docs = {
    get_id(path): read_naf(path)
    for path in DOCS_PATH.glob("*.naf")
}

queries = {
    get_id(path): read_naf(path)
    for path in QUERIES_PATH.glob("*.naf")
}

print("Documentos:", len(docs))
print("Queries:", len(queries))

Documentos: 331
Queries: 35


Veamos un ejemplo

In [54]:
doc_id = next(iter(docs))

print(doc_id)
print(docs[doc_id][:1000])

queries_id = next(iter(queries))


print(queries_id)
print(queries[queries_id][:1000])

d001
William Beaumont and the Human Digestion William Beaumont and the Human Digestion.

William Beaumont: Physiology of digestion Image Source.  On November 21, 1785, US-American surgeon William Beaumont was born. He became best known as “Father of Gastric Physiology” following his research on human digestion. William Beaumont was born in Lebanon, Connecticut and became a physician. He served as a surgeon’s mate in the Army during the War of 1812. He opened a private practice in Plattsburgh, New York, but rejoined the Army as a surgeon in 1819. Beaumont was stationed at Fort Mackinac on Mackinac Island in Michigan in the early 1820s when it existed to protect the interests of the American Fur Company. The fort became the refuge for a wounded 19-year-old French-Canadian fur trader named Alexis St. Martin when a shotgun went off by accident in the American Fur Company store at close range June 6th, 1822. St. Martin’s wound was quite serious because his stomach was perforated and several

### Tokenización

Que vamos a conserver:
- números 
- fechas como 2020-10-25 
- mantener palabras con guion como un único token: state-of-the-art,
- conservar inicialmente caracteres acentuados.

In [55]:
TOKEN_PATTERN = re.compile(
    r"\d{1,4}(?:[/-]\d{1,2}){1,2}"   # fechas
    r"|\d+(?:[.,]\d+)?"              # números
    r"|[^\W\d_]+(?:-[^\W\d_]+)*",    # palabras
    re.UNICODE,
)

In [56]:
def tokenize(text: str) -> list[str]:
    return TOKEN_PATTERN.findall(text)

In [57]:
tokenize("The state-of-the-art model cost 25.5 dollars in 2024-05-10.")

['The',
 'state-of-the-art',
 'model',
 'cost',
 '25.5',
 'dollars',
 'in',
 '2024-05-10']

### Stopwords

In [58]:
STOPWORDS = set(ENGLISH_STOP_WORDS)

In [59]:
def remove_stopwords(tokens: list[str]) -> list[str]:
    return [
        token
        for token in tokens
        if token.lower() not in STOPWORDS
    ]

### Normalizacion

1. Convertir a minúsculas.
2. Quitar tildes/acentos.

In [60]:
def normalize_token(token: str) -> str:
    token = token.lower()

    token = unicodedata.normalize("NFKD", token)

    return "".join(
        char
        for char in token
        if not unicodedata.combining(char)
    )

In [61]:
def normalize(tokens: list[str]) -> list[str]:
    return [normalize_token(token) for token in tokens]

### Stemming

In [62]:
stemmer = SnowballStemmer("english")

In [63]:
def stem_token(token: str) -> str:
    if any(char.isdigit() for char in token):
        return token

    return "-".join(
        stemmer.stem(part)
        for part in token.split("-")
    )

def stem(tokens: list[str]) -> list[str]:
    return [stem_token(token) for token in tokens]

## Pipeline

In [64]:
def preprocess(text: str) -> list[str]:
    tokens = tokenize(text)
    tokens = remove_stopwords(tokens)
    tokens = normalize(tokens)
    tokens = stem(tokens)

    return tokens

In [65]:
### Testeo
text = "The computers are running state-of-the-art algorithms."

print(preprocess(text))

['comput', 'run', 'state-of-the-art', 'algorithm']


In [66]:
def preprocessing_steps(text: str) -> dict[str, list[str]]:
    tokenized = tokenize(text)
    no_stopwords = remove_stopwords(tokenized)
    normalized = normalize(no_stopwords)
    stemmed = stem(normalized)

    return {
        "Tokenización": tokenized,
        "Stopwords": no_stopwords,
        "Normalización": normalized,
        "Stemming": stemmed,
    }

### Tamaño del vocabulario - número total de tokens

In [67]:
def collection_stats(
    collection: dict[str, str],
    name: str,
) -> list[dict]:
    
    stages: dict[str, list[str]] = defaultdict(list)

    for text in collection.values():
        result = preprocessing_steps(text)

        for stage, tokens in result.items():
            stages[stage].extend(tokens)

    return [
        {
            "Colección": name,
            "Paso": stage,
            "Tokens": len(tokens),
            "Vocabulario": len(set(tokens)),
        }
        for stage, tokens in stages.items()
    ]

In [68]:
stats = (
    collection_stats(docs, "Documentos")
    + collection_stats(queries, "Queries")
)

stats_df = pd.DataFrame(stats)

stats_df

,Colección,Paso,Tokens,Vocabulario
0,Documentos,Tokenización,216227,21729
1,Documentos,Stopwords,116556,21255
2,Documentos,Normalización,116556,19086
3,Documentos,Stemming,116556,13994
4,Queries,Tokenización,156,126
5,Queries,Stopwords,116,107
6,Queries,Normalización,116,104
7,Queries,Stemming,116,100


### Documentos procesados

In [69]:
docs_processed = {
    doc_id: preprocess(text)
    for doc_id, text in docs.items()
}

queries_processed = {
    query_id: preprocess(text)
    for query_id, text in queries.items()
}

In [70]:
doc_id = next(iter(docs_processed))

print(doc_id)
print(docs_processed[doc_id][:50])

d001
['william', 'beaumont', 'human', 'digest', 'william', 'beaumont', 'human', 'digest', 'william', 'beaumont', 'physiolog', 'digest', 'imag', 'sourc', 'novemb', '21', '1785', 'us-american', 'surgeon', 'william', 'beaumont', 'born', 'best', 'known', 'father', 'gastric', 'physiolog', 'follow', 'research', 'human', 'digest', 'william', 'beaumont', 'born', 'lebanon', 'connecticut', 'physician', 'serv', 'surgeon', 's', 'mate', 'armi', 'war', '1812', 'open', 'privat', 'practic', 'plattsburgh', 'new', 'york']


# Indice invertido

In [71]:
def build_inverted_index(
    documents: dict[str, list[str]],
) -> dict[str, list[str]]:
    
    index = defaultdict(set)

    for doc_id, tokens in documents.items():
        for token in tokens:
            index[token].add(doc_id)
            #print(token); print(tokens)

    return {
        term: sorted(postings)
        for term, postings in index.items()
    }

In [72]:
inverted_index = build_inverted_index(docs_processed)

print("Términos:", len(inverted_index))

Términos: 13994


In [73]:
term = next(iter(inverted_index))

print(term)
print(inverted_index[term])
print(len(inverted_index[term]))

william
['d001', 'd009', 'd015', 'd028', 'd035', 'd055', 'd056', 'd069', 'd078', 'd088', 'd091', 'd092', 'd095', 'd098', 'd102', 'd106', 'd109', 'd111', 'd129', 'd136', 'd138', 'd147', 'd175', 'd179', 'd180', 'd189', 'd190', 'd191', 'd193', 'd197', 'd212', 'd230', 'd231', 'd241', 'd254', 'd257', 'd266', 'd272', 'd273', 'd274', 'd289', 'd291', 'd294', 'd299', 'd300', 'd309', 'd310', 'd320', 'd323', 'd330']
50


### Posting mas frecuente y menos frecuente

In [74]:
most_frequent = max(
    inverted_index,
    key=lambda term: len(inverted_index[term]),
)

least_frequent = min(
    inverted_index,
    key=lambda term: len(inverted_index[term]),
)

In [75]:
print(
    "Más frecuente:",
    most_frequent,
    len(inverted_index[most_frequent]),
)

print(
    "Menos frecuente:",
    least_frequent,
    len(inverted_index[least_frequent]),
)

Más frecuente: s 321
Menos frecuente: beaumont 1


### Skip pointers

In [76]:
def build_skips(postings: list[str]) -> dict[int, int]:
    p = len(postings)

    if p < 4:
        return {}

    step = int(math.sqrt(p))

    return {
        i: i + step
        for i in range(0, p - step, step)
    }

In [77]:
# testing
postings = list(range(16))

build_skips(postings)

{0: 4, 4: 8, 8: 12}

In [78]:
skip_index = {
    term: build_skips(postings)
    for term, postings in inverted_index.items()
}

In [79]:
print(inverted_index[most_frequent][:20])
print(skip_index[most_frequent])

['d001', 'd002', 'd003', 'd004', 'd005', 'd006', 'd007', 'd009', 'd010', 'd011', 'd012', 'd013', 'd014', 'd015', 'd016', 'd017', 'd018', 'd019', 'd020', 'd021']
{0: 17, 17: 34, 34: 51, 51: 68, 68: 85, 85: 102, 102: 119, 119: 136, 136: 153, 153: 170, 170: 187, 187: 204, 204: 221, 221: 238, 238: 255, 255: 272, 272: 289, 289: 306}


In [80]:
def doc_number(doc_id: str) -> int:
    return int(re.search(r"\d+", doc_id).group())

In [81]:
def intersect(
    p1: list[str],
    p2: list[str],
    s1: dict[int, int],
    s2: dict[int, int],
) -> list[str]:

    result = []

    i = 0
    j = 0

    while i < len(p1) and j < len(p2):

        d1 = doc_number(p1[i])
        d2 = doc_number(p2[j])

        if d1 == d2:
            result.append(p1[i])
            i += 1
            j += 1

        elif d1 < d2:

            if i in s1 and doc_number(p1[s1[i]]) <= d2:
                i = s1[i]
            else:
                i += 1

        else:

            if j in s2 and doc_number(p2[s2[j]]) <= d1:
                j = s2[j]
            else:
                j += 1

    return result

In [82]:
term1 = "comput"
term2 = "model"

result = intersect(
    inverted_index.get(term1, []),
    inverted_index.get(term2, []),
    skip_index.get(term1, {}),
    skip_index.get(term2, {}),
)

result

['d028',
 'd116',
 'd129',
 'd177',
 'd193',
 'd194',
 'd195',
 'd196',
 'd197',
 'd198',
 'd220',
 'd222',
 'd266',
 'd286']

### Tamaño del índice en memoria

In [83]:
def deep_size(obj, seen=None) -> int:
    if seen is None:
        seen = set()

    if id(obj) in seen:
        return 0

    seen.add(id(obj))
    size = sys.getsizeof(obj)

    if isinstance(obj, dict):
        size += sum(
            deep_size(k, seen) + deep_size(v, seen)
            for k, v in obj.items()
        )

    elif isinstance(obj, (list, tuple, set)):
        size += sum(deep_size(x, seen) for x in obj)

    return size

In [84]:
index_size = deep_size(inverted_index)
index_with_skips_size = deep_size(
    (inverted_index, skip_index)
)

print(f"Índice: {index_size / 1024:.2f} KB")
print(f"Índice + skips: {index_with_skips_size / 1024:.2f} KB")

Índice: 2798.30 KB
Índice + skips: 4693.99 KB


### Operador AND con skip pointers

In [85]:
def merge_and(
    p1: list[str],
    p2: list[str],
) -> list[str]:

    s1 = build_skips(p1)
    s2 = build_skips(p2)

    result = []
    i = 0
    j = 0

    while i < len(p1) and j < len(p2):
        d1 = doc_number(p1[i])
        d2 = doc_number(p2[j])

        if d1 == d2:
            result.append(p1[i])
            i += 1
            j += 1

        elif d1 < d2:
            if i in s1 and doc_number(p1[s1[i]]) <= d2:
                i = s1[i]
            else:
                i += 1

        else:
            if j in s2 and doc_number(p2[s2[j]]) <= d1:
                j = s2[j]
            else:
                j += 1

    return result

In [86]:
def merge_or(
    p1: list[str],
    p2: list[str],
) -> list[str]:

    result = []
    i = 0
    j = 0

    while i < len(p1) and j < len(p2):
        d1 = doc_number(p1[i])
        d2 = doc_number(p2[j])

        if d1 == d2:
            result.append(p1[i])
            i += 1
            j += 1

        elif d1 < d2:
            result.append(p1[i])
            i += 1

        else:
            result.append(p2[j])
            j += 1

    result.extend(p1[i:])
    result.extend(p2[j:])

    return result

In [87]:
ALL_DOCS = sorted(
    docs_processed.keys(),
    key=doc_number,
)

In [88]:
def merge_not(
    postings: list[str],
) -> list[str]:

    result = []

    i = 0
    j = 0

    while i < len(ALL_DOCS) and j < len(postings):
        d1 = doc_number(ALL_DOCS[i])
        d2 = doc_number(postings[j])

        if d1 == d2:
            i += 1
            j += 1

        elif d1 < d2:
            result.append(ALL_DOCS[i])
            i += 1

        else:
            j += 1

    result.extend(ALL_DOCS[i:])

    return result

In [104]:
def get_postings(term: str) -> list[str]:

    if term in inverted_index:
        return inverted_index[term]

    tokens = preprocess(term)

    if not tokens:
        return []

    return inverted_index.get(tokens[0], [])

In [90]:
get_postings("telecommunication")

['d060', 'd100', 'd231']

In [91]:
def boolean_tokens(query: str) -> list[str]:
    return re.findall(
        r"\(|\)|AND|OR|NOT|[^\s()]+",
        query,
        flags=re.IGNORECASE,
    )

In [92]:
boolean_tokens(
    "early AND (telecommunication OR methods)"
)

['early', 'AND', '(', 'telecommunication', 'OR', 'methods', ')']

In [93]:
PRECEDENCE = {
    "OR": 1,
    "AND": 2,
    "NOT": 3,
}

In [94]:
def to_postfix(tokens: list[str]) -> list[str]:
    output = []
    operators = []

    for token in tokens:
        op = token.upper()

        if op in PRECEDENCE:

            while (
                operators
                and operators[-1] != "("
                and PRECEDENCE[operators[-1]] >= PRECEDENCE[op]
                and op != "NOT"
            ):
                output.append(operators.pop())

            operators.append(op)

        elif token == "(":
            operators.append(token)

        elif token == ")":
            while operators[-1] != "(":
                output.append(operators.pop())

            operators.pop()

        else:
            output.append(token)

    while operators:
        output.append(operators.pop())

    return output

In [95]:
def boolean_query(query: str) -> list[str]:

    tokens = boolean_tokens(query)
    postfix = to_postfix(tokens)

    stack = []

    for token in postfix:
        op = token.upper()

        if op == "AND":
            b = stack.pop()
            a = stack.pop()
            stack.append(merge_and(a, b))

        elif op == "OR":
            b = stack.pop()
            a = stack.pop()
            stack.append(merge_or(a, b))

        elif op == "NOT":
            a = stack.pop()
            stack.append(merge_not(a))

        else:
            stack.append(get_postings(token))

    return stack[0]

In [96]:
boolean_query(
    "early AND (telecommunication OR methods)"
)

['d024',
 'd034',
 'd035',
 'd048',
 'd052',
 'd060',
 'd068',
 'd069',
 'd076',
 'd085',
 'd091',
 'd093',
 'd100',
 'd109',
 'd113',
 'd118',
 'd122',
 'd124',
 'd142',
 'd194',
 'd201',
 'd212',
 'd215',
 'd221',
 'd231',
 'd233',
 'd235',
 'd244',
 'd263',
 'd279',
 'd281',
 'd282',
 'd285',
 'd295',
 'd305',
 'd330']

In [97]:
boolean_query(
    "early AND NOT methods"
)

['d001',
 'd003',
 'd009',
 'd014',
 'd015',
 'd016',
 'd017',
 'd018',
 'd021',
 'd022',
 'd023',
 'd025',
 'd027',
 'd029',
 'd039',
 'd045',
 'd046',
 'd054',
 'd055',
 'd056',
 'd057',
 'd058',
 'd060',
 'd061',
 'd063',
 'd065',
 'd066',
 'd071',
 'd073',
 'd074',
 'd077',
 'd080',
 'd081',
 'd086',
 'd095',
 'd097',
 'd100',
 'd101',
 'd107',
 'd110',
 'd115',
 'd126',
 'd129',
 'd130',
 'd131',
 'd132',
 'd133',
 'd135',
 'd136',
 'd137',
 'd138',
 'd141',
 'd144',
 'd146',
 'd148',
 'd151',
 'd152',
 'd154',
 'd159',
 'd167',
 'd168',
 'd171',
 'd172',
 'd173',
 'd174',
 'd175',
 'd185',
 'd190',
 'd192',
 'd193',
 'd198',
 'd199',
 'd203',
 'd204',
 'd205',
 'd209',
 'd211',
 'd214',
 'd216',
 'd218',
 'd219',
 'd223',
 'd229',
 'd230',
 'd232',
 'd234',
 'd237',
 'd240',
 'd241',
 'd247',
 'd248',
 'd249',
 'd250',
 'd251',
 'd255',
 'd257',
 'd259',
 'd262',
 'd264',
 'd268',
 'd271',
 'd273',
 'd280',
 'd283',
 'd289',
 'd291',
 'd293',
 'd299',
 'd301',
 'd303',
 'd307',
 

In [98]:
def merge_and_no_skips(
    p1: list[str],
    p2: list[str],
) -> list[str]:

    result = []

    i = 0
    j = 0

    while i < len(p1) and j < len(p2):

        d1 = doc_number(p1[i])
        d2 = doc_number(p2[j])

        if d1 == d2:
            result.append(p1[i])
            i += 1
            j += 1

        elif d1 < d2:
            i += 1

        else:
            j += 1

    return result

### Comparación

In [100]:
def merge_and_no_skips_stats(
    p1: list[str],
    p2: list[str],
):

    result = []

    i = 0
    j = 0
    comparisons = 0

    while i < len(p1) and j < len(p2):

        comparisons += 1

        d1 = doc_number(p1[i])
        d2 = doc_number(p2[j])

        if d1 == d2:
            result.append(p1[i])
            i += 1
            j += 1

        elif d1 < d2:
            i += 1

        else:
            j += 1

    return result, comparisons

In [99]:
def merge_and_skips_stats(
    p1: list[str],
    p2: list[str],
    s1: dict[int, int],
    s2: dict[int, int],
):

    result = []

    i = 0
    j = 0

    comparisons = 0
    skip_jumps = 0
    skipped_postings = 0

    while i < len(p1) and j < len(p2):

        comparisons += 1

        d1 = doc_number(p1[i])
        d2 = doc_number(p2[j])

        if d1 == d2:
            result.append(p1[i])
            i += 1
            j += 1

        elif d1 < d2:

            if i in s1:
                comparisons += 1

                next_i = s1[i]

                if doc_number(p1[next_i]) <= d2:
                    skipped_postings += next_i - i - 1
                    i = next_i
                    skip_jumps += 1
                else:
                    i += 1

            else:
                i += 1

        else:

            if j in s2:
                comparisons += 1

                next_j = s2[j]

                if doc_number(p2[next_j]) <= d1:
                    skipped_postings += next_j - j - 1
                    j = next_j
                    skip_jumps += 1
                else:
                    j += 1

            else:
                j += 1

    return result, comparisons, skip_jumps, skipped_postings

In [101]:
early_term = preprocess("early")[0]
telecommunication_term = preprocess("telecommunication")[0]

p1 = inverted_index.get(early_term, [])
p2 = inverted_index.get(telecommunication_term, [])

s1 = skip_index.get(early_term, {})
s2 = skip_index.get(telecommunication_term, {})

In [102]:
result_no_skips, comparisons_no_skips = merge_and_no_skips_stats(
    p1,
    p2,
)

(
    result_skips,
    comparisons_skips,
    skip_jumps,
    skipped_postings,
) = merge_and_skips_stats(
    p1,
    p2,
    s1,
    s2,
)

print("Postings de early:", len(p1))
print("Postings de telecommunication:", len(p2))

print("\nSin skips")
print("Resultado:", result_no_skips)
print("Comparaciones:", comparisons_no_skips)

print("\nCon skips")
print("Resultado:", result_skips)
print("Comparaciones:", comparisons_skips)
print("Saltos realizados:", skip_jumps)
print("Postings evitados:", skipped_postings)

Postings de early: 158
Postings de telecommunication: 3

Sin skips
Resultado: ['d060', 'd100', 'd231']
Comparaciones: 107

Con skips
Resultado: ['d060', 'd100', 'd231']
Comparaciones: 50
Saltos realizados: 6
Postings evitados: 66


El algoritmo de mezcla sin skip pointers tiene complejidad O(m+n), donde m y n son las longitudes de las listas de postings. La incorporación de skip pointers no modifica la complejidad asintótica del peor caso, que continúa siendo O(m+n), ya que puede ocurrir que ningún salto sea aprovechable. Sin embargo, en casos favorables reduce considerablemente el número de postings examinados al permitir descartar bloques completos de documentos. 

¿Por qué los skips no sirven para OR?

Los skip pointers no ofrecen una ventaja para la operación OR porque la unión debe conservar todos los documentos presentes en cualquiera de las dos listas. A diferencia de AND, no existen bloques de postings que puedan descartarse por no pertenecer al resultado. Saltar un bloque obligaría de todas formas a recuperar sus elementos para agregarlos a la unión, eliminando la ventaja del salto.

### Procesar las 35 consultas

In [105]:
and_results = {}

for query_id in sorted(queries_processed, key=doc_number):

    terms = queries_processed[query_id]

    if not terms:
        result = []
    else:
        query = " AND ".join(terms)
        result = boolean_query(query)

    and_results[query_id] = result

In [106]:
OUTPUT_PATH = Path("BSII-AND-queries_results")

with OUTPUT_PATH.open("w", encoding="utf-8") as f:

    for query_id in sorted(and_results, key=doc_number):

        documents = ",".join(and_results[query_id])

        if documents:
            f.write(f"{query_id} {documents}\n")
        else:
            f.write(f"{query_id}\n")